In [1]:
# Imports

import keras
import numpy as np
import pandas as pd
from hyperopt import STATUS_OK, Trials, fmin, hp, tpe        # For hypeparameter tuning of deep learning model
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import train_test_split
import mlflow
from mlflow.models import infer_signature

ModuleNotFoundError: No module named 'pkg_resources'

In [ ]:
# Load dataset

data = pd.read_csv(
    "https://raw.githubusercontent.com/mlflow/mlflow/master/tests/datasets/winequality-white.csv",
    sep=";"
)

data

,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,pH,sulphates,alcohol,quality
0,7.0,0.27,0.36,20.7,0.045,45.0,170.0,1.00100,3.00,0.45,8.8,6
1,6.3,0.30,0.34,1.6,0.049,14.0,132.0,0.99400,3.30,0.49,9.5,6
2,8.1,0.28,0.40,6.9,0.050,30.0,97.0,0.99510,3.26,0.44,10.1,6
3,7.2,0.23,0.32,8.5,0.058,47.0,186.0,0.99560,3.19,0.40,9.9,6
4,7.2,0.23,0.32,8.5,0.058,47.0,186.0,0.99560,3.19,0.40,9.9,6
...,...,...,...,...,...,...,...,...,...,...,...,...
4893,6.2,0.21,0.29,1.6,0.039,24.0,92.0,0.99114,3.27,0.50,11.2,6
4894,6.6,0.32,0.36,8.0,0.047,57.0,168.0,0.99490,3.15,0.46,9.6,5
4895,6.5,0.24,0.19,1.2,0.041,30.0,111.0,0.99254,2.99,0.46,9.4,6
4896,5.5,0.29,0.30,1.1,0.022,20.0,110.0,0.98869,3.34,0.38,12.8,7


In [ ]:
# Split data into train, test and validation sets

train,test=train_test_split(data, test_size=0.2, random_state=44)

X_train = train.drop(['quality'], axis=1).values
y_train = train[['quality']].values.ravel()

# Test dataset
X_test = test.drop(['quality'], axis=1).values
y_test = test[['quality']].values.ravel()

# Training and validation data
X_train, X_val, y_train, y_val = train_test_split(X_train,y_train,test_size=0.20, random_state=50)

# Signature
signature = infer_signature(X_train,y_train)

In [ ]:
# ANN model trainer function

# Model trainer
def train_model(params,epochs, X_train,y_train,X_val,y_val,X_test,y_test):
    
    # Define model architecture
    mean = np.mean(X_train, axis=0)        # Calculate mean value per independent feature
    var = np.var(X_train,axis=0)           # Calculate variance value per independent feature

    model = keras.Sequential(
        [
            keras.Input([X_train.shape[1]]),
            keras.layers.Normalization(mean=mean, variance=var),
            keras.layers.Dense(64, activation='relu'),
            keras.layers.Dense(1)
        ]
    )

    # Compile model
    model.compile(optimizer=keras.optimizers.SGD(
        learning_rate=params['lr'],
        momentum=params['momentum']
    ),
    loss= "mean_squared_error",
    metrics=[keras.metrics.RootMeanSquaredError()]
    )

    # Train model with ML Flow tracking
    with mlflow.start_run(nested=True):
        model.fit(X_train,y_train,epochs=epochs, batch_size=64, validation_data=(X_val,y_val))

        # Evaluate model
        eval_result = model.evaluate(X_val,y_val, batch_size=64)

        eval_rsme = eval_result[1]

        # Log parameters and results
        mlflow.log_params(params)
        mlflow.log_metric("eval_rmse", eval_rsme)

        # Log model
        mlflow.tensorflow.log_model(model, "ANN model", signature=signature)

        return {"loss": eval_rsme, "status" : STATUS_OK, "model" : model}

In [ ]:
# Objective function

def objective(params):

    # ML Flow will track the parameters and results for each run

    result = train_model(
        params,
        epochs=3,
        X_train=X_train,
        y_train=y_train,
        X_val=X_val,
        y_val=y_val,
        X_test=X_test,
        y_test=y_test
    )
    return result

parameter_space = {
    'lr' : hp.loguniform("lr", np.log(1e-5), np.log(1e-1)),
    "momentum" : hp.uniform("momentum", 0.0, 1.0)
}

In [ ]:
# Hyperparameter Tuning

mlflow.set_experiment("Wine Quality")

with mlflow.start_run():
    # Hyperparameter search
    trials = Trials()
    best = fmin(
        fn= objective,
        space=parameter_space,
        algo=tpe.suggest,
        max_evals=4,
        trials=trials
    )

    # Details of best run
    best_run = sorted(trials.results, key=lambda x:x['loss'])[0]

    # Log best parameters
    mlflow.log_params(best)
    mlflow.log_metric("eval_rmse", best_run['loss'])
    mlflow.tensorflow.log_model(best_run['model'], "model", signature=signature)

    # Print best parameters
    print(f"Best Parameters: {best}")
    print(f"Best eval rmse: {best_run['loss']}")

  0%|          | 0/4 [00:00<?, ?trial/s, best loss=?]

Epoch 1/3                                            

 1/49 ━━━━━━━━━━━━━━━━━━━━ 20s 424ms/step - loss: 36.8640 - root_mean_squared_error: 6.0716
40/49 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - loss: 6.9866 - root_mean_squared_error: 2.4654    
49/49 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - loss: 2.3478 - root_mean_squared_error: 1.5323 - val_loss: 0.6362 - val_root_mean_squared_error: 0.7976

Epoch 2/3                                            

 1/49 ━━━━━━━━━━━━━━━━━━━━ 1s 26ms/step - loss: 0.5753 - root_mean_squared_error: 0.7585
43/49 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - loss: 0.6119 - root_mean_squared_error: 0.7821 
49/49 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.6057 - root_mean_squared_error: 0.7783 - val_loss: 0.5646 - val_root_mean_squared_error: 0.7514

Epoch 3/3                                            

 1/49 ━━━━━━━━━━━━━━━━━━━━ 1s 24ms/step - loss: 0.5755 - root_mean_squared_error: 0.7586
39/49 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - loss: 0.5992 - root_mean_squared_error: 0.7740 
49/

2026/04/24 13:50:53 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.



Epoch 1/3                                                                      

 1/49 ━━━━━━━━━━━━━━━━━━━━ 16s 351ms/step - loss: 41.4402 - root_mean_squared_error: 6.4374
31/49 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 12.4081 - root_mean_squared_error: 3.3530   
49/49 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - loss: 3.6468 - root_mean_squared_error: 1.9097 - val_loss: 0.8580 - val_root_mean_squared_error: 0.9263

Epoch 2/3                                                                      

 1/49 ━━━━━━━━━━━━━━━━━━━━ 1s 26ms/step - loss: 0.6061 - root_mean_squared_error: 0.7785
26/49 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 0.8176 - root_mean_squared_error: 0.9036 
49/49 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.7299 - root_mean_squared_error: 0.8543 - val_loss: 0.6313 - val_root_mean_squared_error: 0.7946

Epoch 3/3                                                                      

 1/49 ━━━━━━━━━━━━━━━━━━━━ 1s 31ms/step - loss: 0.5060 - root_mean_squared_error: 0.7114
34/49 ━━━━━━━━

2026/04/24 13:51:08 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.



...

Epoch 1/3                                                                      

 1/49 ━━━━━━━━━━━━━━━━━━━━ 16s 349ms/step - loss: 38.6241 - root_mean_squared_error: 6.2148
27/49 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 29.3632 - root_mean_squared_error: 5.3971   
49/49 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - loss: 14.2126 - root_mean_squared_error: 3.7700 - val_loss: 3.6318 - val_root_mean_squared_error: 1.9057

Epoch 2/3                                                                      

 1/49 ━━━━━━━━━━━━━━━━━━━━ 1s 25ms/step - loss: 3.3662 - root_mean_squared_error: 1.8347
28/49 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 3.1329 - root_mean_squared_error: 1.7690 
49/49 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 2.5808 - root_mean_squared_error: 1.6065 - val_loss: 2.1919 - val_root_mean_squared_error: 1.4805

Epoch 3/3                                                                      

 1/49 ━━━━━━━━━━━━━━━━━━━━ 1s 27ms/step - loss: 2.6831 - root_mean_squared_error: 1.6380
35/49 ━━━━━━━

2026/04/24 13:51:29 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.



100%|██████████| 4/4 [00:46<00:00, 11.66s/trial, best loss: 0.7533777356147766]

2026/04/24 13:51:38 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.



Best Parameters: {'lr': np.float64(0.05124939368943587), 'momentum': np.float64(0.6543005739146524)}
Best eval rmse: 0.7533777356147766
